# **Dataset Loading**

<a href="https://www.kaggle.com/datasets/shreyasparaj1/loan-prediction-dataset">Dataset Link</a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shreyasparaj1/loan-prediction-dataset")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Pehle file ka naam check karte hain directory mein
files = os.listdir(path)
print("Files in dataset:", files)

# Agar file 'loan_data.csv' hai (aap file ka naam check kar lein)
# Toh is tarah load karein:
df = pd.read_csv(os.path.join(path, files[1]))  # files[1] ko apne file ke index se replace karein

In [ ]:
df.head()

In [ ]:
df.tail()

# **Statistical Analysis**

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.count().unique()

# **Handling Missing Values**

In [ ]:
df.isnull().sum()

In [ ]:
df = df.drop('Loan_ID', axis=1)

In [ ]:
# df["Gender"] = df["Gender"].fillna("Unknown")
df["Gender"] = df["Gender"].fillna(df["Gender"].mode()[0])

In [ ]:
df["Dependents"] = df["Dependents"].fillna(df["Dependents"].mode()[0])

In [ ]:
df["Married"] = df["Married"].fillna(df["Married"].mode()[0])

In [ ]:
# Yeh code har Education level ke liye uska apna mode dhund kar fill karega
df["Self_Employed"] = df.groupby("Education")["Self_Employed"].transform(
    lambda x: x.fillna(x.mode()[0])
)

In [ ]:
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].mean())

In [ ]:
df["Loan_Amount_Term"] = df["Loan_Amount_Term"].fillna(df["Loan_Amount_Term"].mean())

In [ ]:
df["Credit_History"] = df["Credit_History"].fillna(df["Credit_History"].mode()[0])

In [ ]:
df["Total_Income"] = df["ApplicantIncome"] + df["CoapplicantIncome"]

In [ ]:
df.isnull().sum()

In [ ]:
df.head(50)

In [ ]:
df.describe()

# Data Visualization (EDA)

In [ ]:
# df.corr()

## Histogram Data Distribution

In [ ]:
plt.hist(df["Education"], bins=10)
plt.title("Education Distribution")
plt.xlabel("Education")
plt.ylabel("Frequency")
plt.show()

Education histogram is clearly bimodal distirbution

In [ ]:
plt.hist(df["Self_Employed"], bins=10)
plt.title("Self-Employed Distribution")
plt.xlabel("Self-Employed")
plt.ylabel("Frequency")
plt.show()

Self-Employed histogram is clearly bimodal distirbution

In [ ]:
plt.hist(df["LoanAmount"], bins=10)
plt.title("Loan Amount Distribution")
plt.xlabel("Loan Amount")
plt.ylabel("Frequency")
plt.show()

Loan Amount histogram shows skewness 

In [ ]:
plt.hist(df["Total_Income"], bins=10)
plt.title("Total Income Distribution")
plt.xlabel("Total Income")
plt.ylabel("Frequency")
plt.show()

Total Income histogram shows skewness as well as existence of outliers

In [ ]:
plt.hist(df["Credit_History"], bins=10)
plt.title("Credit History Distribution")
plt.xlabel("Credit History")
plt.ylabel("Frequency")
plt.show()

Credit history histogram is clearly bimodal distirbution

## Boxplot (Outlier Detection)

Not applying on `Education` and `Self Employed` because they are categorized as well as are in string objects, not numbers.

In [ ]:
plt.boxplot(df["LoanAmount"])
plt.title("Loan Amount Boxplot")
plt.ylabel("Loan Amount")
plt.show()

In [ ]:
plt.boxplot(df["Total_Income"])
plt.title("Total Income Boxplot")
plt.ylabel("Total Income")
plt.show()

Overall it is clearly visualized that there are alot of outliers. we need to handle them

In [ ]:
plt.boxplot(df["Credit_History"])
plt.title("Credit History Boxplot")
plt.ylabel("Credit History")
plt.show()

Due to bimodal distirbution and less no of zeros causing it to show zeros as outliers

## Scatter Plot

In [ ]:
plt.scatter(df["Total_Income"], df["LoanAmount"], c=df["Credit_History"], alpha=0.5)
plt.title("Total Income vs Loan Amount")
plt.xlabel("Total Income")
plt.ylabel("Loan Amount")
plt.show()

In [ ]:
sns.pairplot(df[["Total_Income", "LoanAmount", "Credit_History"]], hue="Credit_History")

In [ ]:
correlation_matrix = df[["Total_Income", "LoanAmount", "Credit_History"]].corr()
print(correlation_matrix)

# Notes for me
Matrix ko Read Karne ka Tareeqa:
Diagonal Line (Kona se kona): Matrix ke center mein ek line hoti hai jahan values hamesha 1 hoti hain. Iska matlab ye hai ke column khud apna hi 100% perfect positive correlation rakhta hai (e.g., 'Price' ka 'Price' se rishta 1 hai).

Values range (-1 se +1):

- +1 (Perfect Positive): Jaise-jaise ek variable barhta hai, dusra bhi barhta hai.

- -1 (Perfect Negative): Jaise-jaise ek variable barhta hai, dusra kam hota hai.

-  0 (No Correlation): Dono variables ka aapas mein koi ta'aluq nahi hai.

In [ ]:
# %reset -f

In [ ]:
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
skewness = df[["Total_Income", "LoanAmount", "Credit_History"]].skew()
print(skewness)

# **Data Preprocessing**

In [ ]:
df[["Total_Income", "LoanAmount", "Credit_History"]].plot.kde()

In [ ]:
from sklearn.preprocessing import StandardScaler, RobustScaler,LabelEncoder,OneHotEncoder

## Standardization

In [ ]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df[["Total_Income", "LoanAmount",]])
scaled_df = pd.DataFrame(scaled_data, columns=["Total_Income", "LoanAmount"])
scaled_df.plot.kde()

## Log Transformation & Robust Scaler

In [ ]:
df['LoanAmount_log'] = np.log(df['LoanAmount'])
df['Total_Income_log'] = np.log(df['Total_Income'])

In [ ]:
robust = RobustScaler()
robust_data = robust.fit_transform(df[["Total_Income_log", "LoanAmount_log"]])
robust_df = pd.DataFrame(robust_data, columns=["Total_Income", "LoanAmount"])
robust_df.plot.kde()

In [ ]:
# Naye columns ko purane df ke saath jorr dein
df = pd.concat([df, robust_df], axis=1)

# df[["Total_Income_log", "LoanAmount_log"]] = robust.fit_transform(df[["Total_Income_log", "LoanAmount_log"]])

In [ ]:
df.head()

In [ ]:
# Binary columns ke liye Label Encoding

# Purane columns ko update krna k lia...
le = LabelEncoder()

# cols = ['Gender', 'Married', 'Education', 'Self_Employed']

# for i in cols:
#     df[i] = le.fit_transform(df[i])


# if you want custom codes use Map Func()
# # Naye columns banayen (purane waise hi rahenge)

# df['Gender_Code'] = df['Gender'].map({'Male': 1, 'Female': 0})
# df['Married_Code'] = df['Married'].map({'Yes': 1, 'No': 0})
# df['Education_Code'] = df['Education'].map({'Graduate': 1, 'Not Graduate': 0})
# df['Self_Employed_Code'] = df['Self_Employed'].map({'Yes': 1, 'No': 0})
# df['Property_Area_Code'] = df['Property_Area'].map({'Urban': 2, 'Semiurban': 1, 'Rural': 0})

# Updating Actual Columns
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
df['Married'] = df['Married'].map({'Yes': 1, 'No': 0})
df['Education'] = df['Education'].map({'Graduate': 1, 'Not Graduate': 0})
df['Self_Employed'] = df['Self_Employed'].map({'Yes': 1, 'No': 0})
df["Dependents"] = df["Dependents"].replace("3+", 3).astype(int)

# onehotencoding
df = pd.get_dummies(df, columns=["Property_Area"], drop_first=True)

df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})  # required for model training


In [ ]:
df.head(50)

In [ ]:
df.info()

In [ ]:
# Ye khud ba khud naye columns banayega aur purane ko hata dega
# df = pd.get_dummies(df, columns=['Property_Area'], prefix='Area')

# Abhi nai run hoga bcuz humney pehlay he labeling kr li ha
# ohe = OneHotEncoder()
# encoded_data = ohe.fit_transform(df[["Property_Area"]]).toarray()
# encoded_df = pd.DataFrame(encoded_data, columns=ohe.get_feature_names_out(["Property_Area"]))
# df = pd.concat([df, encoded_df], axis=1)

All Preprocessing is Done.

# **Model Training**

In [ ]:
from sklearn.model_selection import train_test_split

## Veritcal Split

In [ ]:
# Features Selection (Raw incomes drop kar diye)
features = ["Gender", "Married", "Dependents", "Education", "Self_Employed", 
            "LoanAmount_log", "Total_Income_log", "Credit_History", 
            "Property_Area_Semiurban", "Property_Area_Urban"]

x = df[features]
y = df["Loan_Status"]

## Horizontal Split

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(x, y , test_size=0.2, random_state=42)

In [ ]:
# # Scaling (Only on numerical continuous features)
# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler()
# cols_to_scale = ["LoanAmount_log", "Total_Income_log"]

# X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
# X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

# X_train.head()

In [ ]:
from sklearn.preprocessing import RobustScaler
robust = RobustScaler()

df['LoanAmount_log'] = np.log(df['LoanAmount_log'])
df['Total_Income_log'] = np.log(df['Total_Income_log'])

cols_to_scale = ["LoanAmount_log", "Total_Income_log"]
    

X_train[cols_to_scale] = robust.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = robust.transform(X_test[cols_to_scale])

X_train.head()

In [ ]:
from sklearn.linear_model import Lasso, Ridge, ElasticNet
rd = Ridge(alpha=0)
rd.fit(X_train, Y_train)

In [ ]:
from sklearn.linear_model import LogisticRegression

# lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

lr = LogisticRegression()
lr.fit(X_train, Y_train)

In [ ]:
lr.coef_

In [ ]:
lr.intercept_

In [ ]:
lr.score(X_test, Y_test) #internally calculates Y_predict from X_test

## Feature Importance

In [ ]:
# Visualize Feature Importance For Logistic Regression
# 1. Coefficients nikalen (Absolute values lein taakay +ive/-ive dono ka asar dikhe)
importances = np.abs(lr.coef_[0]) 
features = x.columns # Ya aapki feature list

# 2. Sort karein
indices = np.argsort(importances)

# 3. Plot karein
plt.figure(figsize=(10, 6))
plt.title('Feature Importance (Based on Coefficients)')
plt.barh(range(len(indices)), importances[indices], color='green', align='center')
plt.yticks(range(len(indices)), [features[i] for i in indices])
plt.xlabel('Absolute Coefficient Value')
plt.show()

# Visualize Feature Importance For Decision Tree
# importances = lr.feature_importances_
# indices = np.argsort(importances)

# plt.figure(figsize=(10, 6))
# plt.title('Feature Importances (Logistic Regression)')
# plt.barh(range(len(indices)), importances[indices], color='green', align='center')
# plt.yticks(range(len(indices)), [features[i] for i in indices])
# plt.xlabel('Relative Importance')
# plt.show()

# **Result Analysis**

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# 2. Accuracy Score
acc = accuracy_score(Y_test, y_pred)
print(f"Overall Accuracy: {acc*100:.2f}%")
print("-" * 30)

In [ ]:
# 3. Classification Report
print("Classification Report: ")
print(classification_report(Y_test, y_pred))

In [ ]:
# 4. Confusion Matrix Heatmap
cm = confusion_matrix(Y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Rejected', 'Approved'], 
            yticklabels=['Rejected', 'Approved'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix Heatmap')
plt.show()

# RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 1. Initialize and Train Random Forest
# We use n_estimators=100 and balanced class weights to maintain the fix for rejected loans
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(X_train, Y_train)

# 2. Evaluate Random Forest
y_pred_rf = rf.predict(X_test)
rf_acc = accuracy_score(Y_test, y_pred_rf)

print(f'Random Forest Accuracy: {rf_acc*100:.2f}%')
print('\nRandom Forest Performance Report:\n', classification_report(Y_test, y_pred_rf))

In [ ]:
# 3. Visualize Feature Importance
importances = rf.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(10, 6))
plt.title('Feature Importances (Random Forest)')
plt.barh(range(len(indices)), importances[indices], color='green', align='center')
plt.yticks(range(len(indices)), [features[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()